In [7]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

In [8]:
def calc_total_variation(p, q):
    p=np.array(p)
    q=np.array(q)
    return 0.5* np.sum(np.abs(p-q))

In [9]:
nosteer_df = pd.read_csv('/Users/nicolemeister/Desktop/STANFORD/distributions/results/human_annotations/Distributional_Alignment_No_Steering_May 21, 2024_18.01.csv')
persona_df = pd.read_csv('/Users/nicolemeister/Desktop/STANFORD/distributions/results/human_annotations/Distributional_Alignment_Persona_Steering_May 21, 2024_19.06.csv')
fewshot_df = pd.read_csv('/Users/nicolemeister/Desktop/STANFORD/distributions/results/human_annotations/Distributional_Alignment_FewShot_Steering_May 21, 2024_18.00.csv')

nosteer_df = nosteer_df.drop(index=[0, 1])
persona_df = persona_df.drop(index=[0, 1])
fewshot_df = fewshot_df.drop(index=[0, 1])

In [10]:
len(nosteer_df), len(persona_df), len(fewshot_df), 

(67, 122, 124)

In [11]:
# col_to_del = ['StartDate',	'EndDate',	'Status',	'IPAddress',	'Progress',	'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId', 'RecipientLastName', 'QDem_Race_6_TEXT', 'Q_DemRepub_6_TEXT',
#               'Q133_1',  'Q133_2', 'Q136']

col_to_keep = ['QDem_Age', 'QDem_Gend', 'QDem_Race', 'Q_DemRepub', 'QDem_Income'] # demographics


# delete the first two rows

In [12]:
data_path ='/Users/nicolemeister/Desktop/STANFORD/distributions/gpt-3.5-turbo-0125/task0/Pew_American_Trends_Panel_disagreement_100/NONE/Democrat.json'
with open(data_path, 'r') as json_file:
    # Load JSON data
    data = json.load(json_file)

In [13]:
qualtricsID_to_qID, qID_to_qualtricsID = {}, {}
for i, qualtricsID in enumerate([1] + list(np.arange(4, 103))):
  col_to_keep.append(str(qualtricsID)+'_Q138')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_1')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_2')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_3')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_4')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_5')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_6')
  col_to_keep.append(str(qualtricsID)+'_Q_NoSteering_7')
  qualtricsID_to_qID[qualtricsID] = list(data.keys())[i]
  qID_to_qualtricsID[list(data.keys())[i]] = qualtricsID

#### In vs Out Group

In [26]:
def compute_one(input_data):   
    # print(np.array(input_data).shape)
    data = input_data
    # convert the list of lists into a full list 
    # for lst in input_data:
    #     print(lst)
    #     data.extend(list(lst))

    num_bootstraps = 1000

    def compute_statistic(sample):
        # todo: weighted average so that nytimes is weighted 50% and opinionqa is weighted 50%
        return np.mean(sample)  

    # Bootstrapping process
    bootstrap_statistics = []
    for _ in range(num_bootstraps):
        bootstrap_sample = np.random.choice(data, size=len(data), replace=True)
        statistic = compute_statistic(bootstrap_sample)
        bootstrap_statistics.append(statistic)

    # 95% confidence interval
    confidence_level = 0.95
    alpha = (1 - confidence_level) / 2
    lower_percentile = alpha * 100
    upper_percentile = (1 - alpha) * 100
    lower_bound = np.percentile(bootstrap_statistics, lower_percentile)
    upper_bound = np.percentile(bootstrap_statistics, upper_percentile)
    
    return np.mean(bootstrap_statistics), (upper_bound-lower_bound)/2

def compute_tv_human_annotations(df, nosteer=True, k=4, ingroup=False, outgroup=False, ingroup_name = 'Democrat'):
  if ingroup: print('in group')
  if outgroup: print('out group')
  dem_group_to_dem_mapping = {
                              'POLPARTY': ['Democrat', 'Republican'],
                              'SEX': ['Male', 'Female'],
                              'RACE': ['Black', 'White']
                              }

  data_path ='/Users/nicolemeister/Desktop/STANFORD/distributions/gpt-3.5-turbo-0125/task0/Pew_American_Trends_Panel_disagreement_100/NONE/Democrat.json'
  with open(data_path, 'r') as json_file:
    # Load JSON data
    data = json.load(json_file)

  dem_to_qualtrics_map = {'Democrat': 'Q_NoSteering', 'Republican': 'Q135', 'Male':'Q136', 'Female': 'Q137', 'Black': 'Q138', 'White': 'Q139'}
  dem_to_demgroup = {'Democrat': 'POLPARTY', 'Republican': 'POLPARTY', 'Male':'SEX', 'Female': 'SEX', 'Black': 'RACE', 'White': 'RACE'}

  tv_across_groups, hv_across_groups, expected_across_groups = [], [], []

  if ingroup: 
    list_of_groups = [ingroup_name]
  elif outgroup:
    list_of_groups = dem_group_to_dem_mapping[dem_to_demgroup[ingroup_name]].copy()
    list_of_groups.remove(ingroup_name)
  else: list_of_groups = list(dem_to_demgroup.keys())
  # print(ingroup_name, list_of_groups)

  for dem in list_of_groups:
    num_responses, all_tvs = [], []
    for qID in list(data.keys()):
      qualtricsID = qID_to_qualtricsID[qID]
      cols = []
      if nosteer: qualtrics_str = 'Q_NoSteering'
      else: 
        qualtrics_str = dem_to_qualtrics_map[dem]
      cols.append(str(qualtricsID)+'_{}_1'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_2'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_3'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_4'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_5'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_6'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_7'.format(qualtrics_str))

      df_cleaned = df[cols].dropna()[:k]
      num_responses.append(len(df_cleaned))
      if len(df_cleaned)>0: 
        expected_results_str = 'expected_results_{}_{}'.format(dem_to_demgroup[dem], dem)
        MC_options = list(data[qID][expected_results_str].keys())
        human_values = np.mean(np.array(df_cleaned, dtype=np.float64), axis=0)[:len(data[qID][expected_results_str].keys())] * 0.01
        human_results = dict(zip(MC_options, human_values))

        gt_results = np.array(list(data[qID][expected_results_str].values()))
        # print(human_values, gt_results)
        tv = calc_total_variation(human_values, gt_results)
        all_tvs.append(tv)
        hv_across_groups.append(human_values)
        expected_across_groups.append(gt_results)
    tv_across_groups.extend(all_tvs)
    mean, bs = compute_one(all_tvs)
    print("{}: {:.3f} +/- {:.3f}".format(dem, mean, bs))

  mean, bs = compute_one(tv_across_groups)
  return mean, bs, tv_across_groups# , hv_across_groups, expected_across_groups



def compute_tv_human_annotations_avg(df, nosteer=True, k=4, ingroup=False, outgroup=False, ingroup_name = 'Democrat'):
  dem_group_to_dem_mapping = {
                              'POLPARTY': ['Democrat', 'Republican'],
                              'SEX': ['Male', 'Female'],
                              'RACE': ['Black', 'White']
                              }

  data_path ='/Users/nicolemeister/Desktop/STANFORD/distributions/gpt-3.5-turbo-0125/task0/Pew_American_Trends_Panel_disagreement_100/NONE/Democrat.json'
  with open(data_path, 'r') as json_file:
    # Load JSON data
    data = json.load(json_file)

  dem_to_qualtrics_map = {'Democrat': 'Q_NoSteering', 'Republican': 'Q135', 'Male':'Q136', 'Female': 'Q137', 'Black': 'Q138', 'White': 'Q139'}
  dem_to_demgroup = {'Democrat': 'POLPARTY', 'Republican': 'POLPARTY', 'Male':'SEX', 'Female': 'SEX', 'Black': 'RACE', 'White': 'RACE'}

  tv_across_groups, hv_across_groups, expected_across_groups = [], [], []

  if ingroup: 
    list_of_groups = [ingroup_name]
  elif outgroup:
    list_of_groups = dem_group_to_dem_mapping[dem_to_demgroup[ingroup_name]].copy()
    list_of_groups.remove(ingroup_name)
  else: list_of_groups = list(dem_to_demgroup.keys())
  # print(ingroup_name, list_of_groups)

  for dem in list_of_groups:
    num_responses, all_tvs = [], []
    for qID in list(data.keys()):
      qualtricsID = qID_to_qualtricsID[qID]
      cols = []
      if nosteer: qualtrics_str = 'Q_NoSteering'
      else: 
        qualtrics_str = dem_to_qualtrics_map[dem]
      cols.append(str(qualtricsID)+'_{}_1'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_2'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_3'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_4'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_5'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_6'.format(qualtrics_str))
      cols.append(str(qualtricsID)+'_{}_7'.format(qualtrics_str))

      df_cleaned = df[cols].dropna()[:k]
      num_responses.append(len(df_cleaned))
      if len(df_cleaned)>0: 
        expected_results_str = 'expected_results_{}_{}'.format(dem_to_demgroup[dem], dem)
        MC_options = list(data[qID][expected_results_str].keys())
        human_values = np.mean(np.array(df_cleaned, dtype=np.float64), axis=0)[:len(data[qID][expected_results_str].keys())] * 0.01
        human_results = dict(zip(MC_options, human_values))

        try: gt_results = (np.array(list(data[qID]['expected_results_POLPARTY_Democrat'].values())) + np.array(list(data[qID]['expected_results_POLPARTY_Republican'].values())) + np.array(list(data[qID]['expected_results_SEX_Male'].values())) + np.array(list(data[qID]['expected_results_SEX_Female'].values())) + np.array(list(data[qID]['expected_results_RACE_White'].values())) + np.array(list(data[qID]['expected_results_RACE_Black'].values())))/6
        except: gt_results=np.zeros(len(human_values))
        # print(human_values, gt_results)
        tv = calc_total_variation(human_values, gt_results)
        all_tvs.append(tv)
        hv_across_groups.append(human_values)
        expected_across_groups.append(gt_results)
    tv_across_groups.extend(all_tvs)
    
    # print(len(all_tvs))

  mean, bs = compute_one(tv_across_groups)
  return mean, bs, tv_across_groups# , hv_across_groups, expected_across_groups

In [17]:
# Purpose: comparing how democrats simulate democrats, to how democrats simulate 
# # HOW TO DO IN GROUP
# filter the input df for a particular group, pass in the group name
dem_to_qualtrics  = {'Democrat': 'Q_DemRepub', 'Republican': 'Q_DemRepub', 'Male':'QDem_Gend', 'Female': 'QDem_Gend', 'Black': 'QDem_Race', 'White': 'QDem_Race'}
qualtricsdem_to_dem = {'White': 'White or Caucasian American', 'Black': 'Black or African American', 'Democrat': 'Democrat',  'Republican':  'Republican', 'Male':'Male', 'Female': 'Female'}


print("Persona")
all_tvs_ingroup, all_tvs_outgroup = [], []
for dem in dem_to_qualtrics.keys():
    filtered_df = persona_df[persona_df[dem_to_qualtrics[dem]]==qualtricsdem_to_dem[dem]]
    _, _, ingroup = compute_tv_human_annotations(filtered_df, nosteer=False, ingroup=True, outgroup=False, ingroup_name = dem)
    _, _, outgroup = compute_tv_human_annotations(filtered_df, nosteer=False, ingroup=False, outgroup=True, ingroup_name = dem)
    all_tvs_ingroup.extend(ingroup)
    all_tvs_outgroup.extend(outgroup)
        
mean_in, bs_in = compute_one(all_tvs_ingroup)
mean_out, bs_out = compute_one(all_tvs_outgroup)
print("Persona In group: {:4f} +/- {:4f}".format(mean_in, bs_in ))
print("Persona Out group: {:4f} +/- {:4f}".format(mean_out, bs_out))


print('Few Shot')
all_tvs_ingroup, all_tvs_outgroup = [], []
for dem in dem_to_qualtrics.keys():
    filtered_df = fewshot_df[fewshot_df[dem_to_qualtrics[dem]]==qualtricsdem_to_dem[dem]]
    _, _, ingroup = compute_tv_human_annotations(filtered_df, nosteer=False, ingroup=True, outgroup=False, ingroup_name = dem)
    _, _, outgroup = compute_tv_human_annotations(filtered_df, nosteer=False, ingroup=False, outgroup=True, ingroup_name = dem)
    all_tvs_ingroup.extend(ingroup)
    all_tvs_outgroup.extend(outgroup)


mean_in, bs_in = compute_one(all_tvs_ingroup)
mean_out, bs_out = compute_one(all_tvs_outgroup)
print("Few Shot In group: {:4f} +/- {:4f}".format(mean_in, bs_in ))
print("Few Shot Out group: {:4f} +/- {:4f}".format(mean_out, bs_out))


Persona
in group
Democrat: 0.237 +/- 0.022
out group
Republican: 0.318 +/- 0.027
in group
Republican: 0.371 +/- 0.036
out group
Democrat: 0.368 +/- 0.043
in group
Male: 0.321 +/- 0.037
out group
Female: 0.310 +/- 0.032
in group
Female: 0.250 +/- 0.024
out group
Male: 0.295 +/- 0.025
in group
Black: 0.404 +/- 0.055
out group
White: 0.400 +/- 0.050
in group
White: 0.273 +/- 0.022
out group
Black: 0.283 +/- 0.030
Persona In group: 0.298111 +/- 0.013296
Persona Out group: 0.321810 +/- 0.013442
Few Shot
in group
Democrat: 0.257 +/- 0.027
out group
Republican: 0.261 +/- 0.028
in group
Republican: 0.338 +/- 0.044
out group
Democrat: 0.326 +/- 0.044
in group
Male: 0.284 +/- 0.028
out group
Female: 0.279 +/- 0.033
in group
Female: 0.254 +/- 0.024
out group
Male: 0.253 +/- 0.028
in group
Black: 0.389 +/- 0.047
out group
White: 0.323 +/- 0.034
in group
White: 0.233 +/- 0.020
out group
Black: 0.261 +/- 0.026
Few Shot In group: 0.282902 +/- 0.013627
Few Shot Out group: 0.277675 +/- 0.013052


### Storing the data

In [30]:
nosteer, persona, fewshot= [], [], []
nosteer_mean, nosteer_bs, nosteer_data = compute_tv_human_annotations(nosteer_df, nosteer=True, ingroup=False, outgroup=False)
persona_mean, persona_bs, persona_data = compute_tv_human_annotations(persona_df, nosteer=False, ingroup=False, outgroup=False)
fewshot_mean, fewshot_bs, fewshot_data = compute_tv_human_annotations(fewshot_df, nosteer=False, ingroup=False, outgroup=False)

print("No Steer: {:4f} +/- {:4f}".format(nosteer_mean, nosteer_bs))
print("Persona: {:4f} +/- {:4f}".format(persona_mean, persona_bs))
print("Few Shot: {:4f} +/- {:4f}".format(fewshot_mean, fewshot_bs))

human_annotation_data = {'nosteer': nosteer_data, 'persona': persona_data, 'fewshot': fewshot_data}


import json

# Specify the file name
filename = '/Users/nicolemeister/Desktop/STANFORD/distributions/results/human_annotations/OQA_human_tv_data.json'

# Writing to a JSON file
with open(filename, 'w') as file:
    json.dump(human_annotation_data, file, indent=4)

Democrat: 0.301 +/- 0.024
Republican: 0.271 +/- 0.023
Male: 0.236 +/- 0.019
Female: 0.253 +/- 0.021
Black: 0.280 +/- 0.024
White: 0.242 +/- 0.020
Democrat: 0.244 +/- 0.026
Republican: 0.299 +/- 0.025
Male: 0.269 +/- 0.024
Female: 0.243 +/- 0.021
Black: 0.284 +/- 0.027
White: 0.269 +/- 0.022
Democrat: 0.249 +/- 0.025
Republican: 0.253 +/- 0.027
Male: 0.241 +/- 0.022
Female: 0.240 +/- 0.024
Black: 0.276 +/- 0.027
White: 0.240 +/- 0.022
No Steer: 0.264267 +/- 0.009764
Persona: 0.267895 +/- 0.010118
Few Shot: 0.249792 +/- 0.010653
